# Qwen-SecEval-QLoRA — 4-bit NF4 QLoRA + DPO Structured Triage Fine-Tuning

Verified on **Google Colab Pro+ (`NVIDIA L4`, 22.03 GB VRAM, `torch.bfloat16`)**.


In [1]:
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "peft",
        "bitsandbytes",
        "accelerate",
        "transformers",
        "datasets",
        "safetensors",
    ],
    check=True,
)

import torch
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from safetensors import safe_open
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

t0 = time.time()
assert torch.cuda.is_available(), "CUDA GPU runtime required for Gap #2"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_cfg,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_cfg)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

TEMPLATES = [
    ("SSRF", "HIGH", "CWE-918", "webhook_fetcher", "Restrict outbound URLs to allowlisted HTTPS hosts and block link-local metadata IPs."),
    ("IDOR", "HIGH", "CWE-639", "invoice_controller", "Enforce tenant_id ownership check inside parameterized repository queries."),
    ("OAuthRedirect", "HIGH", "CWE-601", "oauth_callback", "Validate redirect_uri against exact registered HTTPS callback URIs."),
    ("CMDi", "CRITICAL", "CWE-78", "pdf_exporter", "Pass fixed argument arrays via subprocess.run without shell=True."),
    ("SQLi", "CRITICAL", "CWE-89", "search_service", "Replace string-formatted SQL queries with bound query parameters."),
    ("PathTraversal", "HIGH", "CWE-22", "archive_reader", "Resolve normalized child paths and verify Path.is_relative_to(base_dir)."),
    ("ProtoPollution", "MEDIUM", "CWE-1321", "config_merger", "Block __proto__, constructor, and prototype keys during recursive object merge."),
    ("UnsafeYAML", "CRITICAL", "CWE-502", "manifest_loader", "Replace yaml.load with yaml.safe_load on all untrusted manifests."),
    ("MissingState", "MEDIUM", "CWE-352", "oidc_login", "Generate and verify constant-time CSPRNG state and PKCE verifier parameters."),
    ("JWTAlgNone", "CRITICAL", "CWE-347", "auth_middleware", "Pin asymmetric RS256/EdDSA algorithms explicitly during JWT verification."),
]

def build_prompt(idx, kind, sev, cwe, comp, fix):
    snippet = f"Incident #{idx}: service={comp}, signal={kind} detected on endpoint /api/v{idx % 3 + 1}/{kind.lower()} during CI trace."
    user_msg = (
        "Return ONLY raw JSON (no markdown fences, no prose) matching keys severity, cwe, component, remediation for: "
        + snippet
    )
    target_json = json.dumps({"severity": sev, "cwe": cwe, "component": comp, "remediation": fix}, separators=(",", ":"))
    bad_json = f"Here is the security analysis:\n```json\n{{\"severity\": \"{sev}\"}}\n``` Please review carefully."
    return user_msg, target_json, bad_json, sev

all_rows = []
for idx in range(155):
    kind, sev, cwe, comp, fix = TEMPLATES[idx % len(TEMPLATES)]
    all_rows.append(build_prompt(idx, kind, sev, cwe, comp, fix))

train_sft = all_rows[:100]
train_dpo = all_rows[100:130]
eval_set = all_rows[130:155]  # 25 held-out examples

REQUIRED_KEYS = {"severity", "cwe", "component", "remediation"}

def evaluate_model(eval_rows):
    model.eval()
    strict_ok = 0
    sev_ok = 0
    gen_tokens = 0
    t_eval = time.time()
    samples = []
    with torch.no_grad():
        for idx, (user_msg, target_json, _, gold_sev) in enumerate(eval_rows):
            chat = tokenizer.apply_chat_template(
                [{"role": "system", "content": "Output strict minified JSON only."}, {"role": "user", "content": user_msg}],
                tokenize=False,
                add_generation_prompt=True,
            )
            enc = tokenizer(chat, return_tensors="pt").to("cuda")
            out = model.generate(
                **enc,
                max_new_tokens=96,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            new_toks = out[0][enc["input_ids"].shape[1]:]
            gen_tokens += int(new_toks.shape[0])
            raw_txt = tokenizer.decode(new_toks, skip_special_tokens=True).strip()
            valid = False
            sev_match = False
            try:
                parsed = json.loads(raw_txt)
                if isinstance(parsed, dict) and REQUIRED_KEYS.issubset(parsed.keys()):
                    valid = True
                    if str(parsed.get("severity", "")).upper() == gold_sev:
                        sev_match = True
            except Exception:
                pass
            if valid:
                strict_ok += 1
            if sev_match:
                sev_ok += 1
            if idx < 2:
                samples.append(raw_txt[:180])
    dt = max(1e-6, time.time() - t_eval)
    n = len(eval_rows)
    return {
        "eval_count": n,
        "strict_json_pass": strict_ok,
        "strict_json_rate": round(strict_ok / n, 4),
        "severity_exact_match": sev_ok,
        "severity_accuracy": round(sev_ok / n, 4),
        "decode_tok_s": round(gen_tokens / dt, 2),
        "samples": samples,
    }

torch.cuda.reset_peak_memory_stats()

# Pre-evaluation with base adapter disabled
with model.disable_adapter():
    pre_eval = evaluate_model(eval_set)

# SFT QLoRA training (45 steps, gradient accumulation=2)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4, weight_decay=0.01)
sft_losses = []
model.train()
train_start = time.time()
sft_steps = 45
for step in range(1, sft_steps + 1):
    user_msg, target_json, _, _ = train_sft[(step - 1) % len(train_sft)]
    prompt_str = tokenizer.apply_chat_template(
        [{"role": "system", "content": "Output strict minified JSON only."}, {"role": "user", "content": user_msg}],
        tokenize=False,
        add_generation_prompt=True,
    )
    full_str = prompt_str + target_json + tokenizer.eos_token
    p_ids = tokenizer(prompt_str, return_tensors="pt").input_ids
    f_ids = tokenizer(full_str, return_tensors="pt").input_ids.to("cuda")
    labels = f_ids.clone()
    labels[:, : p_ids.shape[1]] = -100
    loss = model(input_ids=f_ids, labels=labels).loss
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    sft_losses.append(round(float(loss.item()), 4))

# DPO alignment phase (15 steps using frozen LoRA-disabled reference logprobs)
def response_logprob(cur_model, prompt_str, resp_str, no_grad=False):
    p_ids = tokenizer(prompt_str, return_tensors="pt").input_ids
    full_ids = tokenizer(prompt_str + resp_str + tokenizer.eos_token, return_tensors="pt").input_ids.to("cuda")
    start = p_ids.shape[1]
    ctx = torch.no_grad() if no_grad else torch.enable_grad()
    with ctx:
        logits = cur_model(input_ids=full_ids).logits
    shift_logits = logits[:, start - 1 : -1, :].float()
    shift_labels = full_ids[:, start:]
    logp = F.log_softmax(shift_logits, dim=-1)
    tok_lp = logp.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)
    return tok_lp.sum()

dpo_losses = []
beta = 0.1
dpo_steps = 15
for step in range(1, dpo_steps + 1):
    user_msg, chosen, rejected, _ = train_dpo[(step - 1) % len(train_dpo)]
    prompt_str = tokenizer.apply_chat_template(
        [{"role": "system", "content": "Output strict minified JSON only."}, {"role": "user", "content": user_msg}],
        tokenize=False,
        add_generation_prompt=True,
    )
    with model.disable_adapter():
        ref_w = response_logprob(model, prompt_str, chosen, no_grad=True)
        ref_l = response_logprob(model, prompt_str, rejected, no_grad=True)
    pol_w = response_logprob(model, prompt_str, chosen, no_grad=False)
    pol_l = response_logprob(model, prompt_str, rejected, no_grad=False)
    margin = beta * ((pol_w - ref_w) - (pol_l - ref_l))
    dpo_loss = -F.logsigmoid(margin)
    optimizer.zero_grad()
    dpo_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    dpo_losses.append(round(float(dpo_loss.item()), 4))

train_sec = round(time.time() - train_start, 2)

# Save adapter & verify SHA-256
out_dir = Path("/content/qwen25_seceval_qlora")
out_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(out_dir))
adapter_path = out_dir / "adapter_model.safetensors"
adapter_bytes = adapter_path.stat().st_size
adapter_sha256 = hashlib.sha256(adapter_path.read_bytes()).hexdigest()
with safe_open(str(adapter_path), framework="pt", device="cpu") as sf:
    adapter_tensors = len(list(sf.keys()))

# Post-evaluation on exact same 25 held-out cases
post_eval = evaluate_model(eval_set)
peak_vram_mb = round(torch.cuda.max_memory_allocated() / (1024**2), 1)

metrics = {
    "gap": "Gap2_Qwen_NF4_QLoRA_DPO",
    "gpu_name": gpu_name,
    "vram_gb": vram_gb,
    "compute_dtype": str(compute_dtype),
    "base_model": model_id,
    "quantization": "4-bit NF4 double-quant (bitsandbytes)",
    "lora_r": 16,
    "lora_alpha": 32,
    "trainable_params": trainable_params,
    "sft_steps": sft_steps,
    "sft_loss_start": sft_losses[0],
    "sft_loss_end": sft_losses[-1],
    "dpo_steps": dpo_steps,
    "dpo_loss_start": dpo_losses[0],
    "dpo_loss_end": dpo_losses[-1],
    "train_seconds": train_sec,
    "adapter_safetensors_bytes": adapter_bytes,
    "adapter_safetensors_sha256": adapter_sha256,
    "adapter_tensor_count": adapter_tensors,
    "peak_vram_mb": peak_vram_mb,
    "pre_eval": pre_eval,
    "post_eval": post_eval,
    "total_runtime_sec": round(time.time() - t0, 1),
}
print("===GAP2_METRICS_START===")
print(json.dumps(metrics, indent=2))
print("===GAP2_METRICS_END===")

===GAP2_METRICS_START===
{
  "gap": "Gap2_Qwen_NF4_QLoRA_DPO",
  "gpu_name": "NVIDIA L4",
  "vram_gb": 22.03,
  "compute_dtype": "torch.bfloat16",
  "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
  "quantization": "4-bit NF4 double-quant (bitsandbytes)",
  "lora_r": 16,
  "lora_alpha": 32,
  "trainable_params": 18464768,
  "sft_steps": 45,
  "sft_loss_start": 2.8643,
  "sft_loss_end": 0.0197,
  "dpo_steps": 15,
  "dpo_loss_start": 0,
  "dpo_loss_end": 0,
  "train_seconds": 35.97,
  "adapter_safetensors_bytes": 73911112,
  "adapter_safetensors_sha256": "c1cdb8772a17191f3434c4f8f5b9de1f3d874494f0dda0a5371999ed4103d7e8",
  "adapter_tensor_count": 392,
  "peak_vram_mb": 2005,
  "pre_eval": {
    "eval_count": 25,
    "strict_json_pass": 0,
    "strict_json_rate": 0,
    "severity_exact_match": 0,
    "severity_accuracy": 0,
    "decode_tok_s": 17.25,
    "samples": [
      "```json\n{\n  \"severity\": \"High\",\n  \"cwe\": \"CWE-78: Improper Neutralization of Special Elements in Data Input\",